In [64]:
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

In [65]:
def month_encoded(df: pd.DataFrame):
    df['month_sin'] = np.sin(2 * np.pi * df['Месяц'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['Месяц'] / 12)
    return df

In [66]:
def robust_encoded(df_train: pd.DataFrame, df_test: pd.DataFrame):
    rs = RobustScaler()
    df_train[[
        'Численность населения',
        'Количество домохозяйств',
        'Трафик пеший, в час',
        'Трафик авто, в час'
    ]] = rs.fit_transform(
        df_train[[
            'Численность населения',
            'Количество домохозяйств',
            'Трафик пеший, в час',
            'Трафик авто, в час'
        ]]
    )
    df_test[[
        'Численность населения',
        'Количество домохозяйств',
        'Трафик пеший, в час',
        'Трафик авто, в час'
    ]] = rs.transform(
        df_test[[
            'Численность населения',
            'Количество домохозяйств',
            'Трафик пеший, в час',
            'Трафик авто, в час'
        ]]
    )

    return df_train, df_test

In [67]:
def pca_applied(df_train: pd.DataFrame, df_test: pd.DataFrame):
    counts = [
        'Маркетплейсы, доставки, постаматы (100 м)',
        'Медицинские уч. и аптеки (300 м)',
        'Школы (300 м)',
        'Остановки (300 м)',
        'Продуктовые магазины (500 м)',
        'Пятерочки (500 м)'
    ]
    pca_param = df_train[counts]

    pca_param_train, pca_param_test = train_test_split(pca_param, test_size=0.3, random_state=598)
    pipeline = Pipeline([('scaler', StandardScaler()),('pca', PCA(n_components=4, random_state=598))])

    pipeline.fit(pca_param_train)

    pca_train = pipeline.transform(df_train.iloc[pca_param_train.index][counts])
    pca_train_valid = pipeline.transform(df_train.iloc[pca_param_test.index][counts])
    pca_test = pipeline.transform(df_test[counts])

    print(f'Доля объясненной дисперсии: {sum(pipeline.named_steps['pca'].explained_variance_ratio_)}')

    pca_train_df = pd.DataFrame(
        np.vstack([pca_train, pca_train_valid]),
        columns=['pca_1', 'pca_2', 'pca_3', 'pca_4'],
        index=np.concatenate([pca_param_train.index, pca_param_test.index])
    ).sort_index()
    pca_test_df = pd.DataFrame(
        pca_test,
        columns=['pca_1', 'pca_2', 'pca_3', 'pca_4']
    )

    df_train = pd.concat([df_train, pca_train_df], axis=1)
    df_test = pd.concat([df_test, pca_test_df], axis=1)

    return df_train, df_test

In [68]:
def target_encoder_traffic_avg_bill(df_train: pd.DataFrame, df_test: pd.DataFrame):
    df_train['Населенный пункт_fold_traffic'] = np.nan
    df_train['Регион_fold_traffic'] = np.nan
    df_train['Населенный пункт_fold_average_bill'] = np.nan
    df_train['Регион_fold_average_bill'] = np.nan

    encoders_traffic = []
    encoders_average_bill = []

    gkf = GroupKFold(n_splits=5)

    for train_id, validation_id in gkf.split(df_train, groups=df_train['new_id']):
    
        X_train = df_train.iloc[train_id][['Населенный пункт', 'Регион']]
        X_validation = df_train.iloc[validation_id][['Населенный пункт', 'Регион']]
        y_train_traffic = df_train.iloc[train_id]['Трафик']
        y_train_average_bill = df_train.iloc[train_id]['Средний чек']
        
        TE_traffic = TargetEncoder(cols = ['Населенный пункт', 'Регион'], smoothing = 20)
        TE_traffic.fit(X_train, y_train_traffic)
        val_traffic = TE_traffic.transform(X_validation)
        encoders_traffic.append(TE_traffic)
        
        df_train.iloc[validation_id, df_train.columns.get_loc('Населенный пункт_fold_traffic')] = val_traffic['Населенный пункт']
        df_train.iloc[validation_id, df_train.columns.get_loc('Регион_fold_traffic')] = val_traffic['Регион']
        
        TE_average_bill = TargetEncoder(cols = ['Населенный пункт', 'Регион'], smoothing = 20)
        TE_average_bill.fit(X_train, y_train_average_bill)
        val_average_bill = TE_average_bill.transform(X_validation)
        encoders_average_bill.append(TE_average_bill)
        
        df_train.iloc[validation_id, df_train.columns.get_loc('Населенный пункт_fold_average_bill')] = val_average_bill['Населенный пункт']
        df_train.iloc[validation_id, df_train.columns.get_loc('Регион_fold_average_bill')] = val_average_bill['Регион']
    
    encoded_sum_traffic_df_test = None
    encoded_sum_average_bill_df_test = None

    for traffic_encoder in encoders_traffic:
        encoded = traffic_encoder.transform(df_test[['Населенный пункт', 'Регион']])
        if encoded_sum_traffic_df_test is None:
            encoded_sum_traffic_df_test = encoded
        else:
            encoded_sum_traffic_df_test += encoded
    
    for average_bill_encoder in encoders_average_bill:
        encoded = average_bill_encoder.transform(df_test[['Населенный пункт', 'Регион']])
        if encoded_sum_average_bill_df_test is None:
            encoded_sum_average_bill_df_test = encoded
        else:
            encoded_sum_average_bill_df_test += encoded
    
    df_test_traffic_encoded =  encoded_sum_traffic_df_test / len(encoders_traffic)
    df_test_average_bill_encoded = encoded_sum_average_bill_df_test / len(encoders_average_bill)

    df_train_traffic = df_train.drop(
        columns=[
            'Средний чек',
            'Населенный пункт_fold_average_bill',
            'Регион_fold_average_bill'
        ]
    )
    df_train_average_bill = df_train.drop(
        columns=[
            'Трафик',
            'Населенный пункт_fold_traffic',
            'Регион_fold_traffic'
        ]
    )

    df_test_traffic = df_test.copy()
    df_test_traffic[['Населенный пункт_fold_traffic', 'Регион_fold_traffic']] = df_test_traffic_encoded

    df_test_average_bill = df_test.copy()
    df_test_average_bill[['Населенный пункт_fold_average_bill', 'Регион_fold_average_bill']] = df_test_average_bill_encoded

    return df_train_traffic, df_train_average_bill, df_test_traffic, df_test_average_bill

In [69]:
def ohe_encoded(df_train: pd.DataFrame, df_test: pd.DataFrame):
    categorical_features = ['Дата открытия, категориальный', 'Торговая площадь, категориальный']

    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    df_train_ohe = ohe.fit_transform(df_train[categorical_features]).astype(int)
    df_test_ohe = ohe.transform(df_test[categorical_features]).astype(int)

    df_train[ohe.get_feature_names_out(categorical_features)] = df_train_ohe
    df_test[ohe.get_feature_names_out(categorical_features)] = df_test_ohe

    return df_train, df_test

In [73]:
df_train = pd.read_excel('../data/Х5.xlsx')
df_test = pd.read_csv('../data/result_test_df.csv')
df_test.head()

,Полный адрес помещения,"Торговая площадь, вещественный",url,Месяц,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,1,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,2,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,3,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0
3,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,4,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0
4,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,5,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0


In [71]:
df_train = month_encoded(df_train)
df_test = month_encoded(df_test)

df_train, df_test = robust_encoded(df_train, df_test)
df_train, df_test = pca_applied(df_train, df_test)
df_train_traffic, df_train_average_bill, df_test_traffic, df_test_average_bill = target_encoder_traffic_avg_bill(df_train, df_test)
df_train_traffic, df_test_traffic = ohe_encoded(df_train_traffic, df_test_traffic)
df_train_average_bill, df_test_average_bill = ohe_encoded(df_train_average_bill, df_test_average_bill)

df_train_traffic.head()

Доля объясненной дисперсии: 0.8537810195273763


,new_id,Месяц,Трафик,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час",...,pca_4,Населенный пункт_fold_traffic,Регион_fold_traffic,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний"
0,0,10,59662,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,-0.182274,-0.533564,-0.54306,...,-1.215406,57225.767865,57387.962262,0,0,1,0,0,0,1
1,0,5,56674,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,-0.182274,-0.533564,-0.54306,...,-1.215406,57225.767865,57387.962262,0,0,1,0,0,0,1
2,0,1,51488,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,-0.182274,-0.533564,-0.54306,...,-1.215406,57225.767865,57387.962262,0,0,1,0,0,0,1
3,0,6,56693,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,-0.182274,-0.533564,-0.54306,...,-1.215406,57225.767865,57387.962262,0,0,1,0,0,0,1
4,0,7,58128,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,-0.182274,-0.533564,-0.54306,...,-1.215406,57225.767865,57387.962262,0,0,1,0,0,0,1


In [72]:
df_train_traffic.to_csv('../data/df_train_traffic_encoded.csv', index=False, encoding='utf-8')
df_test_traffic.to_csv('../data/df_test_traffic_encoded.csv', index=False, encoding='utf-8')
df_train_average_bill.to_csv('../data/df_train_average_bill_encoded.csv', index=False, encoding='utf-8')
df_test_average_bill.to_csv('../data/df_test_average_bill_encoded.csv', index=False, encoding='utf-8')